# LeetCode 347 — Top K Frequent Elements bằng Python

Link đề: https://leetcode.com/problems/top-k-frequent-elements/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table (bảng băm), Heap / Priority Queue (hàng đợi ưu tiên), Bucket Sort (sắp xếp thùng), Quickselect (chọn nhanh), Counting (đếm)

Skeleton LeetCode:

```python
class Solution:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với bộ kiểm tra độc lập), nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng số nguyên `nums` và số `k`, trả về **`k` giá trị xuất hiện nhiều lần nhất** (thứ tự tùy ý; đề đảm bảo đáp án là **duy nhất**).
2. Nắm các công cụ Python: `dict` / `collections.Counter` để **đếm tần suất**, `sorted(..., key=...)` với `lambda`, `heapq` (min-heap), danh sách các danh sách làm **thùng** (`[[] for _ in range(m)]`), và cắt lát `a[:k]`.
3. Đi từ **đếm từng giá trị bằng `list.count`** → **`Counter` + sắp xếp** → **min-heap kích thước `k`** → **bucket sort theo tần suất `O(n)`** → **Quickselect trung bình `O(n)`**.
4. Hiểu mấu chốt: **bước 1 luôn là đếm tần suất bằng hash map**, bước 2 là bài toán "chọn `k` phần tử lớn nhất theo tần suất" mà **không cần sắp xếp toàn bộ**; và tần suất của một giá trị luôn nằm trong `[1, n]`, nên có thể dùng chỉ số mảng làm khóa (bucket).
5. Tránh các lỗi kinh điển: sắp xếp **ngược chiều**, dùng `[[]] * n` (các thùng dùng chung một list), đặt `(giá trị, tần suất)` thay vì `(tần suất, giá trị)` trong heap, và tạo thùng thiếu một ô (tần suất tối đa là `n`, cần `n + 1` ô).

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums` và một số nguyên `k`. Hãy trả về **`k` phần tử xuất hiện nhiều lần nhất**. Có thể trả về theo **bất kỳ thứ tự** nào.

**Ví dụ 1**

- Input: `nums = [1,1,1,2,2,3]`, `k = 2`
- Output: `[1,2]`
- Giải thích: `1` xuất hiện `3` lần, `2` xuất hiện `2` lần, `3` xuất hiện `1` lần; hai giá trị nhiều nhất là `1` và `2`.

**Ví dụ 2**

- Input: `nums = [1]`, `k = 1`
- Output: `[1]`

**Ví dụ 3**

- Input: `nums = [1,2,1,2,1,2,3,1,3,2]`, `k = 2`
- Output: `[1,2]`
- Giải thích: `1` và `2` đều xuất hiện `4` lần, `3` xuất hiện `2` lần.

**Ràng buộc:**

- `1 <= nums.length <= 10^5`
- `-10^4 <= nums[i] <= 10^4`
- `k` nằm trong khoảng `[1, số phần tử khác nhau của mảng]`
- Đề **đảm bảo đáp án là duy nhất** (không có hòa tần suất làm "tranh" vị trí thứ `k`).

**Follow-up:** thuật toán phải có độ phức tạp thời gian **tốt hơn `O(n log n)`**, với `n` là kích thước mảng.

In [1]:
from typing import List

examples = [
    ([1, 1, 1, 2, 2, 3], 2, [1, 2]),
    ([1], 1, [1]),
    ([1, 2, 1, 2, 1, 2, 3, 1, 3, 2], 2, [1, 2]),
]
for nums, k, expected in examples:
    print("nums =", nums, "k =", k, "-> kỳ vọng (mọi thứ tự)", expected)

nums = [1, 1, 1, 2, 2, 3] k = 2 -> kỳ vọng (mọi thứ tự) [1, 2]
nums = [1] k = 1 -> kỳ vọng (mọi thứ tự) [1]
nums = [1, 2, 1, 2, 1, 2, 3, 1, 3, 2] k = 2 -> kỳ vọng (mọi thứ tự) [1, 2]


## 3. Kiến thức nền

### 3.1. Đếm tần suất bằng `dict` và `Counter`

**Tần suất** của một giá trị là số lần nó xuất hiện. Dùng `dict`: khóa là giá trị, giá trị là số lần. Cách viết tay: `count[x] = count.get(x, 0) + 1` (`get(x, 0)` trả `0` nếu chưa có khóa). `collections.Counter` làm việc đó trong một dòng, và có `most_common(k)` trả `k` cặp `(giá trị, tần suất)` nhiều nhất.

In [2]:
from collections import Counter

nums = [1, 1, 1, 2, 2, 3]
count = {}
for x in nums:
    count[x] = count.get(x, 0) + 1       # chưa có thì lấy 0 rồi cộng 1
print(count)                              # {1: 3, 2: 2, 3: 1}

c = Counter(nums)
print(c)                                  # Counter({1: 3, 2: 2, 3: 1})
print(c[1], c[99])                        # 3 0   (khóa vắng mặt cho 0, không lỗi)
print(c.most_common(2))                   # [(1, 3), (2, 2)]
print(list(c.items()), list(c.keys()), list(c.values()))

{1: 3, 2: 2, 3: 1}
Counter({1: 3, 2: 2, 3: 1})
3 0
[(1, 3), (2, 2)]
[(1, 3), (2, 2), (3, 1)] [1, 2, 3] [3, 2, 1]


### 3.2. Sắp xếp theo khóa tùy ý: `sorted(..., key=..., reverse=...)`

`sorted(iterable)` trả list mới đã sắp **tăng dần**. `key=` là hàm cho mỗi phần tử một "điểm" để so sánh; `lambda x: count[x]` là hàm vô danh trả tần suất của `x`. Thêm `reverse=True` để sắp **giảm dần**. Đây là nguồn của lỗi "đảo chiều" rất hay gặp.

In [3]:
c = Counter([1, 1, 1, 2, 2, 3])
print(sorted(c))                                          # [1, 2, 3]  (theo giá trị khóa)
print(sorted(c, key=lambda x: c[x]))                      # [3, 2, 1]  tăng dần theo tần suất
print(sorted(c, key=lambda x: c[x], reverse=True))        # [1, 2, 3]  giảm dần theo tần suất
print(sorted(c, key=lambda x: -c[x]))                     # cũng giảm dần (đổi dấu)
print(sorted(c, key=lambda x: -c[x])[:2])                 # [1, 2]  hai phần tử đầu = top 2

[1, 2, 3]
[3, 2, 1]
[1, 2, 3]
[1, 2, 3]
[1, 2]


### 3.3. Min-heap với `heapq`

**Heap** là cấu trúc cho phép lấy phần tử **nhỏ nhất** rất nhanh. Trong Python, `heapq` biến một `list` thành **min-heap**: `heappush(h, x)` thêm `x`, `heappop(h)` lấy và bỏ phần tử nhỏ nhất, `h[0]` xem phần tử nhỏ nhất (không bỏ); mỗi thao tác `O(log m)` với `m` là kích thước heap. Phần tử là **tuple** thì so sánh theo từng thành phần từ trái sang phải: `(cnt, val)` so `cnt` trước.

Ý tưởng cho bài này: giữ một heap **chỉ chứa `k` phần tử tốt nhất**; phần tử nhỏ nhất của heap (`h[0]`) là "ứng viên bị loại đầu tiên".

In [4]:
import heapq

h = []
for x in [5, 1, 4, 2, 3]:
    heapq.heappush(h, x)
print(h[0], len(h))                       # 1 5   (nhỏ nhất ở đỉnh)
print([heapq.heappop(h) for _ in range(len(h))])      # [1, 2, 3, 4, 5]

pairs = []
for cnt, val in [(3, 1), (2, 2), (1, 3)]:
    heapq.heappush(pairs, (cnt, val))
print(pairs[0])                           # (1, 3)  tần suất nhỏ nhất ở đỉnh
print(heapq.heapreplace(pairs, (5, 9)), pairs[0])     # lấy (1,3) ra, đẩy (5,9) vào, rồi xem đỉnh mới

1 5
[1, 2, 3, 4, 5]
(1, 3)
(1, 3) (2, 2)


### 3.4. Danh sách các thùng: `[[] for _ in range(m)]` và cái bẫy `[[]] * m`

**Thùng (bucket)** là một list nhỏ; nhiều thùng xếp thành một list lớn, thùng `i` chứa các phần tử "có điểm `i`". Phải tạo bằng `[[] for _ in range(m)]` để mỗi thùng là **một list riêng**. Viết `[[]] * m` tạo `m` ô cùng trỏ vào **cùng một list** — thêm vào một thùng là thêm vào tất cả.

In [5]:
good = [[] for _ in range(3)]
good[0].append("a")
print(good)                               # [['a'], [], []]

bad = [[]] * 3
bad[0].append("a")
print(bad)                                # [['a'], ['a'], ['a']]  <- cả ba là MỘT list
print(bad[0] is bad[1])                   # True

[['a'], [], []]
[['a'], ['a'], ['a']]
True


### 3.5. Cắt lát (slicing) và duyệt ngược

`a[:k]` lấy `k` phần tử đầu (nếu `a` ngắn hơn `k` thì lấy hết, không lỗi). `range(n, 0, -1)` đếm ngược `n, n-1, ..., 1`: ta sẽ dùng để duyệt các thùng từ tần suất **cao** xuống **thấp**. `res.extend(lst)` nối cả `lst` vào cuối `res`.

In [6]:
a = [10, 20, 30, 40]
print(a[:2], a[:10])                      # [10, 20] [10, 20, 30, 40]
print(list(range(5, 0, -1)))              # [5, 4, 3, 2, 1]
res = []
res.extend([7, 8])
res.extend([])
print(res)                                # [7, 8]

[10, 20] [10, 20, 30, 40]
[5, 4, 3, 2, 1]
[7, 8]


### 3.6. Chia mảng theo "mốc" (pivot): nền tảng của Quickselect

Chọn một giá trị mốc `p` rồi chia các phần tử thành ba nhóm: **lớn hơn** `p`, **bằng** `p`, **nhỏ hơn** `p`. Nếu ta cần `k` phần tử lớn nhất và nhóm "lớn hơn" có đúng nhiều hơn hay ít hơn `k`, ta biết nên tìm tiếp **ở nhóm nào** mà không phải sắp xếp tất cả. Đây là ý tưởng Quickselect; viết bằng list comprehension rất dễ đọc.

In [7]:
items = [5, 2, 8, 2, 9, 1]
p = 5
bigger = [x for x in items if x > p]
equal = [x for x in items if x == p]
smaller = [x for x in items if x < p]
print(bigger, equal, smaller)             # [8, 9] [5] [2, 2, 1]
# cần 3 phần tử lớn nhất: 2 phần tử ở 'bigger' + 1 phần tử ở 'equal'
print(bigger + equal[: 3 - len(bigger)])  # [8, 9, 5]

[8, 9] [5] [2, 2, 1]
[8, 9, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [1,1,1,2,2,3]`, `k = 2`. Việc đầu tiên **một người sẽ làm** là gạch đếm từng giá trị:

| giá trị | số lần xuất hiện |
|---|---|
| 1 | 3 |
| 2 | 2 |
| 3 | 1 |

Sau đó "chọn `k = 2` giá trị có số lần lớn nhất": nhìn vào cột thứ hai, lấy `3` và `2` lớn nhất ⇒ giá trị `1` và `2`.

Điều đáng nhận ra: **ta không cần xếp hạng toàn bộ** mọi giá trị; chỉ cần biết "ai thuộc nhóm top `k`". Và con số tần suất luôn là số nguyên trong `[1, n]` (`n = len(nums)`), nên ta có thể **dùng tần suất làm chỉ số** của một mảng thùng.

In [8]:
nums, k = [1, 1, 1, 2, 2, 3], 2
count = Counter(nums)
print("tần suất:", dict(count))
by_freq = {}
for value, freq in count.items():
    by_freq.setdefault(freq, []).append(value)         # gom các giá trị theo tần suất
print("gom theo tần suất:", by_freq)                    # {3: [1], 2: [2], 1: [3]}
result = []
for freq in sorted(by_freq, reverse=True):
    result.extend(by_freq[freq])
print("lấy từ cao xuống thấp:", result[:k])            # [1, 2]

tần suất: {1: 3, 2: 2, 3: 1}
gom theo tần suất: {3: [1], 2: [2], 1: [3]}
lấy từ cao xuống thấp: [1, 2]


## 5. Cách 1 — Brute Force / baseline: `list.count` cho từng giá trị

**Ý tưởng.** Lấy các giá trị **khác nhau** của mảng (`set(nums)`). Với mỗi giá trị `v`, **đếm** bằng `nums.count(v)` (quét toàn mảng). Tạo danh sách cặp `(số lần, v)`, **sắp xếp giảm dần** rồi lấy `k` giá trị đầu. Rất dễ nghĩ, bám sát đề.

In [9]:
def top_k_bruteforce(nums: List[int], k: int) -> List[int]:
    pairs = []
    for v in set(nums):                    # mỗi giá trị khác nhau đúng một lần
        pairs.append((nums.count(v), v))   # nums.count(v) quét toàn bộ nums: O(n)
    pairs.sort(reverse=True)               # tần suất lớn trước
    return [v for _, v in pairs[:k]]


for nums, k, expected in examples:
    print(sorted(top_k_bruteforce(nums, k)), "(kỳ vọng", sorted(expected), ")")

[1, 2] (kỳ vọng [1, 2] )
[1] (kỳ vọng [1] )
[1, 2] (kỳ vọng [1, 2] )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`, `u` là số giá trị khác nhau. Mỗi `count` tốn `O(n)`, có `u` lần ⇒ `O(n · u)`; sắp xếp `O(u log u)`. Trường hợp xấu `u ≈ n` cho **`O(n²)`**. Space `O(u)`.

**Nút thắt:** mỗi `nums.count(v)` quét lại cả mảng. Ta chỉ cần **một lượt duyệt** để đếm mọi giá trị cùng lúc bằng `dict`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — đếm một lần.** Duyệt `nums` một lượt, `count[x] += 1` ⇒ `O(n)` có được tần suất của mọi giá trị.

**Bước 2 — chọn `k` giá trị có tần suất lớn nhất.** Đây là bài toán "top `k`" kinh điển với bốn công cụ:

| Công cụ | Ý tưởng | Time |
|---|---|---|
| Sắp xếp | xếp `u` giá trị theo tần suất, lấy `k` đầu | `O(u log u)` |
| Min-heap cỡ `k` | duy trì `k` giá trị tốt nhất, loại kẻ yếu nhất | `O(u log k)` |
| Bucket sort | thùng `f` chứa các giá trị có tần suất `f`; duyệt từ `n` xuống `1` | `O(n)` |
| Quickselect | chia theo mốc, chỉ đi tiếp ở phía chứa ranh giới top `k` | `O(u)` trung bình |

Follow-up đòi **tốt hơn `O(n log n)`**: sắp xếp thuần có thể là `O(n log n)` khi `u ≈ n`, nên ta cần heap (khi `k` nhỏ), **bucket** hoặc **Quickselect** để thật sự tránh sắp xếp toàn bộ.

In [10]:
nums, k = [1, 1, 1, 2, 2, 3], 2
count = Counter(nums)                                   # bước 1: đếm
buckets = [[] for _ in range(len(nums) + 1)]            # thùng 0..n
for value, freq in count.items():
    buckets[freq].append(value)
print("buckets:", buckets)                              # [[], [3], [2], [1], [], [], []]
answer = []
for freq in range(len(nums), 0, -1):                    # từ tần suất cao xuống thấp
    answer.extend(buckets[freq])
    if len(answer) >= k:
        break
print(answer[:k])                                        # [1, 2]

buckets: [[], [3], [2], [1], [], [], []]
[1, 2]


## 7. Các cách tiếp theo

### Cách 2 — `Counter` + sắp xếp theo tần suất

**Giải thích từng đoạn.**

1. `count = Counter(nums)`: đếm `O(n)`.
2. `sorted(count, key=count.get, reverse=True)`: sắp các **khóa** theo tần suất giảm dần (`count.get` là hàm trả tần suất; `reverse=True` để giảm dần).
3. Lấy `k` phần tử đầu: `[:k]`.

Đây là cách ngắn nhất, đủ tốt khi không đòi hỏi follow-up.

In [11]:
def top_k_sort(nums: List[int], k: int) -> List[int]:
    count = Counter(nums)
    return sorted(count, key=count.get, reverse=True)[:k]


for nums, k, expected in examples:
    print(sorted(top_k_sort(nums, k)), "(kỳ vọng", sorted(expected), ")")

[1, 2] (kỳ vọng [1, 2] )
[1] (kỳ vọng [1] )
[1, 2] (kỳ vọng [1, 2] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from collections import Counter
from typing import List

class SolutionSort:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        count = Counter(nums)
        return sorted(count, key=count.get, reverse=True)[:k]


print(SolutionSort().topKFrequent([1, 1, 1, 2, 2, 3], 2))     # [1, 2]

[1, 2]


**Độ phức tạp Cách 2:** đếm `O(n)`, sắp xếp `O(u log u)` ⇒ **Time `O(n + u log u)`** (xấu nhất `O(n log n)`), **Space `O(u)`**.

### Cách 3 — Min-heap kích thước `k`

**Ý tưởng.** Duyệt từng cặp `(giá trị, tần suất)`. Giữ một **min-heap** các cặp `(tần suất, giá trị)` với **tối đa `k` phần tử**:

- Nếu heap chưa đủ `k`: đẩy vào.
- Nếu đủ `k` mà tần suất mới **lớn hơn** đỉnh heap (`heap[0]`, tần suất nhỏ nhất trong `k` người đang giữ): **đuổi** đỉnh, nhận người mới (`heapreplace`).
- Cuối cùng heap chứa đúng `k` giá trị tần suất cao nhất.

Vì sao dùng **min**-heap cho top **lớn nhất**? Vì ta cần nhanh chóng biết và loại **người yếu nhất** trong nhóm `k`.

In [13]:
def top_k_heap(nums: List[int], k: int) -> List[int]:
    count = Counter(nums)
    heap = []                                      # chứa (tần suất, giá trị), tối đa k phần tử
    for value, freq in count.items():
        if len(heap) < k:
            heapq.heappush(heap, (freq, value))
        elif freq > heap[0][0]:                    # mạnh hơn người yếu nhất đang giữ
            heapq.heapreplace(heap, (freq, value))
    return [value for _, value in heap]


for nums, k, expected in examples:
    print(sorted(top_k_heap(nums, k)), "(kỳ vọng", sorted(expected), ")")

[1, 2] (kỳ vọng [1, 2] )
[1] (kỳ vọng [1] )
[1, 2] (kỳ vọng [1, 2] )


In [14]:
# Code nộp LeetCode cho Cách 3
import heapq
from collections import Counter
from typing import List

class SolutionHeap:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        count = Counter(nums)
        heap = []
        for value, freq in count.items():
            if len(heap) < k:
                heapq.heappush(heap, (freq, value))
            elif freq > heap[0][0]:
                heapq.heapreplace(heap, (freq, value))
        return [value for _, value in heap]


print(sorted(SolutionHeap().topKFrequent([1, 1, 1, 2, 2, 3], 2)))    # [1, 2]

[1, 2]


**Độ phức tạp Cách 3:** đếm `O(n)`; mỗi trong `u` giá trị tốn `O(log k)` ⇒ **Time `O(n + u log k)`** (tốt hơn `O(n log n)` khi `k` nhỏ), **Space `O(u + k)`** (bảng đếm và heap).

Mẹo ngắn gọn có sẵn: `heapq.nlargest(k, count.keys(), key=count.get)` cho cùng kết quả (nội bộ dùng heap cỡ `k`).

### Cách 4 — Bucket sort theo tần suất (`O(n)`)

**Ý tưởng.** Tần suất của mọi giá trị nằm trong `[1, n]`, nên tạo `n + 1` thùng; thùng `f` chứa **các giá trị có tần suất đúng `f`**. Sau khi đếm, đặt mỗi giá trị vào thùng của tần suất nó, rồi **duyệt thùng từ `n` xuống `1`**, gom giá trị cho đến khi đủ `k`. Không cần so sánh/sắp xếp.

**Giải thích từng đoạn.**

1. `count = Counter(nums)`.
2. `buckets = [[] for _ in range(len(nums) + 1)]` (cần `n + 1` ô vì tần suất có thể bằng đúng `n`, khi mọi phần tử giống nhau).
3. `buckets[freq].append(value)` cho mọi cặp.
4. `for freq in range(len(nums), 0, -1)`: nối các giá trị của thùng vào `res` và **dừng khi `len(res) >= k`**.

In [15]:
def top_k_bucket(nums: List[int], k: int) -> List[int]:
    count = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]       # buckets[f] = các giá trị có tần suất f
    for value, freq in count.items():
        buckets[freq].append(value)
    res = []
    for freq in range(len(nums), 0, -1):               # từ tần suất cao xuống thấp
        res.extend(buckets[freq])
        if len(res) >= k:
            break
    return res[:k]


for nums, k, expected in examples:
    print(sorted(top_k_bucket(nums, k)), "(kỳ vọng", sorted(expected), ")")

[1, 2] (kỳ vọng [1, 2] )
[1] (kỳ vọng [1] )
[1, 2] (kỳ vọng [1, 2] )


In [16]:
# Code nộp LeetCode cho Cách 4
from collections import Counter
from typing import List

class SolutionBucket:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        count = Counter(nums)
        buckets = [[] for _ in range(len(nums) + 1)]
        for value, freq in count.items():
            buckets[freq].append(value)
        res = []
        for freq in range(len(nums), 0, -1):
            res.extend(buckets[freq])
            if len(res) >= k:
                break
        return res[:k]


print(SolutionBucket().topKFrequent([1, 2, 1, 2, 1, 2, 3, 1, 3, 2], 2))     # [1, 2] (mọi thứ tự)

[1, 2]


**Độ phức tạp Cách 4:** đếm `O(n)`, đặt vào thùng `O(u)`, duyệt thùng `O(n + u)` ⇒ **Time `O(n)`**, **Space `O(n)`** (các thùng). Đây là cách **thật sự** `O(n)` không phụ thuộc `k`.

### Cách 5 — Quickselect (chọn nhanh, trung bình `O(n)`)

**Ý tưởng.** Với danh sách các giá trị khác nhau `items` (điểm số = tần suất), muốn `k` phần tử có điểm cao nhất:

1. Chọn **ngẫu nhiên** một mốc `p` (tần suất của một phần tử bất kỳ).
2. Chia `items` thành `bigger` (tần suất `> p`), `equal` (`== p`), `smaller` (`< p`).
3. Nếu `k <= len(bigger)`: đáp án nằm hoàn toàn trong `bigger` ⇒ lặp lại với `items = bigger`.
4. Nếu `k <= len(bigger) + len(equal)`: lấy hết `bigger` cộng `k - len(bigger)` phần tử đầu của `equal` ⇒ **xong**.
5. Ngược lại: nhận hết `bigger + equal` vào kết quả, giảm `k` đi tương ứng, rồi tìm tiếp trong `smaller`.

Mỗi vòng, `items` co lại cỡ một nửa (mốc ngẫu nhiên) nên tổng công việc `n + n/2 + ... = O(n)` trung bình. Chia **ba nhóm** (thay vì hai) giúp không bị chậm khi có nhiều tần suất bằng nhau.

In [17]:
import random


def top_k_quickselect(nums: List[int], k: int) -> List[int]:
    count = Counter(nums)
    items = list(count.keys())
    res = []
    while True:
        pivot = count[random.choice(items)]                       # mốc ngẫu nhiên
        bigger = [x for x in items if count[x] > pivot]
        equal = [x for x in items if count[x] == pivot]
        smaller = [x for x in items if count[x] < pivot]
        if k <= len(bigger):
            items = bigger                                        # ranh giới top k nằm trong 'bigger'
        elif k <= len(bigger) + len(equal):
            return res + bigger + equal[: k - len(bigger)]        # đủ k: xong
        else:
            res += bigger + equal                                 # nhận hết hai nhóm này
            k -= len(bigger) + len(equal)
            items = smaller


for nums, k, expected in examples:
    print(sorted(top_k_quickselect(nums, k)), "(kỳ vọng", sorted(expected), ")")

[1, 2] (kỳ vọng [1, 2] )
[1] (kỳ vọng [1] )
[1, 2] (kỳ vọng [1, 2] )


In [18]:
# Code nộp LeetCode cho Cách 5
import random
from collections import Counter
from typing import List

class SolutionQuickselect:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        count = Counter(nums)
        items = list(count.keys())
        res = []
        while True:
            pivot = count[random.choice(items)]
            bigger = [x for x in items if count[x] > pivot]
            equal = [x for x in items if count[x] == pivot]
            smaller = [x for x in items if count[x] < pivot]
            if k <= len(bigger):
                items = bigger
            elif k <= len(bigger) + len(equal):
                return res + bigger + equal[: k - len(bigger)]
            else:
                res += bigger + equal
                k -= len(bigger) + len(equal)
                items = smaller


print(sorted(SolutionQuickselect().topKFrequent([1, 1, 1, 2, 2, 3], 2)))     # [1, 2]

[1, 2]


**Độ phức tạp Cách 5:** **Time `O(n)` trung bình** (đếm `O(n)` + các vòng chia `O(u + u/2 + ...)`; trường hợp cực xấu `O(u²)` rất hiếm do mốc ngẫu nhiên), **Space `O(u)`** (các danh sách con).

## 8. TRACE từng bước

Ta trace Cách 4 (bucket) trên `nums = [1,1,1,2,2,3]`, `k = 2`; Cách 3 (heap) trên `nums = [4,1,-1,2,-1,2,3]`, `k = 2`; và Cách 5 (Quickselect) với hạt giống cố định để thấy từng vòng chia.

In [19]:
def trace_bucket(nums, k):
    count = Counter(nums)
    print(f"nums = {nums}, k = {k}")
    print("  tần suất:", dict(count))
    buckets = [[] for _ in range(len(nums) + 1)]
    for value, freq in count.items():
        buckets[freq].append(value)
        print(f"  đặt {value} vào thùng {freq}")
    print("  các thùng:", buckets)
    res = []
    for freq in range(len(nums), 0, -1):
        print(f"  xét thùng {freq}: {buckets[freq]}, res trước = {res}")
        res.extend(buckets[freq])
        if len(res) >= k:
            print(f"  đã đủ {k} phần tử, dừng; res = {res}")
            break
    return res[:k]


assert sorted(trace_bucket([1, 1, 1, 2, 2, 3], 2)) == [1, 2]

nums = [1, 1, 1, 2, 2, 3], k = 2
  tần suất: {1: 3, 2: 2, 3: 1}
  đặt 1 vào thùng 3
  đặt 2 vào thùng 2
  đặt 3 vào thùng 1
  các thùng: [[], [3], [2], [1], [], [], []]
  xét thùng 6: [], res trước = []
  xét thùng 5: [], res trước = []
  xét thùng 4: [], res trước = []
  xét thùng 3: [1], res trước = []
  xét thùng 2: [2], res trước = [1]
  đã đủ 2 phần tử, dừng; res = [1, 2]


In [20]:
def trace_heap(nums, k):
    count = Counter(nums)
    print(f"nums = {nums}, k = {k}, tần suất = {dict(count)}")
    heap = []
    for value, freq in count.items():
        if len(heap) < k:
            heapq.heappush(heap, (freq, value))
            print(f"  ({freq},{value}): heap chưa đủ {k} -> đẩy vào, heap = {sorted(heap)}")
        elif freq > heap[0][0]:
            out = heapq.heapreplace(heap, (freq, value))
            print(f"  ({freq},{value}): {freq} > đỉnh -> đuổi {out}, nhận ({freq},{value}), heap = {sorted(heap)}")
        else:
            print(f"  ({freq},{value}): {freq} <= đỉnh {heap[0][0]} -> bỏ qua, heap = {sorted(heap)}")
    return [v for _, v in heap]


assert sorted(trace_heap([4, 1, -1, 2, -1, 2, 3], 2)) == [-1, 2]

nums = [4, 1, -1, 2, -1, 2, 3], k = 2, tần suất = {4: 1, 1: 1, -1: 2, 2: 2, 3: 1}
  (1,4): heap chưa đủ 2 -> đẩy vào, heap = [(1, 4)]
  (1,1): heap chưa đủ 2 -> đẩy vào, heap = [(1, 1), (1, 4)]
  (2,-1): 2 > đỉnh -> đuổi (1, 1), nhận (2,-1), heap = [(1, 4), (2, -1)]
  (2,2): 2 > đỉnh -> đuổi (1, 4), nhận (2,2), heap = [(2, -1), (2, 2)]
  (1,3): 1 <= đỉnh 2 -> bỏ qua, heap = [(2, -1), (2, 2)]


In [21]:
def trace_quickselect(nums, k, seed=7):
    rng = random.Random(seed)
    count = Counter(nums)
    items = list(count.keys())
    res = []
    print(f"tần suất = {dict(count)}, k = {k}")
    round_no = 0
    while True:
        round_no += 1
        pivot = count[rng.choice(items)]
        bigger = [x for x in items if count[x] > pivot]
        equal = [x for x in items if count[x] == pivot]
        smaller = [x for x in items if count[x] < pivot]
        print(f"  vòng {round_no}: items={items}, mốc tần suất={pivot}, bigger={bigger}, equal={equal}, smaller={smaller}, k={k}")
        if k <= len(bigger):
            print("    k <= len(bigger): tìm tiếp trong bigger")
            items = bigger
        elif k <= len(bigger) + len(equal):
            ans = res + bigger + equal[: k - len(bigger)]
            print(f"    đủ k: kết quả = {ans}")
            return ans
        else:
            res += bigger + equal
            k -= len(bigger) + len(equal)
            print(f"    nhận hết bigger+equal, res = {res}, còn cần k = {k}, tìm tiếp trong smaller")
            items = smaller


assert sorted(trace_quickselect([1, 1, 1, 2, 2, 2, 2, 3, 3, 4], 2)) == [1, 2]

tần suất = {1: 3, 2: 4, 3: 2, 4: 1}, k = 2
  vòng 1: items=[1, 2, 3, 4], mốc tần suất=2, bigger=[1, 2], equal=[3], smaller=[4], k=2
    k <= len(bigger): tìm tiếp trong bigger
  vòng 2: items=[1, 2], mốc tần suất=3, bigger=[2], equal=[1], smaller=[], k=2
    đủ k: kết quả = [2, 1]


Bảng trace bucket cho `nums = [1,1,1,2,2,3]`, `k = 2`:

| bước | hành động | trạng thái |
|---|---|---|
| đếm | `Counter` | `{1: 3, 2: 2, 3: 1}` |
| đặt thùng | `buckets[3] = [1]`, `buckets[2] = [2]`, `buckets[1] = [3]` | `[[], [3], [2], [1], [], [], []]` |
| freq = 6, 5, 4 | thùng rỗng | `res = []` |
| freq = 3 | thêm `[1]` | `res = [1]` (chưa đủ `k = 2`) |
| freq = 2 | thêm `[2]` | `res = [1, 2]` đủ ⇒ dừng |

Đáp án `[1, 2]`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 4 (bucket sort)**: `O(n)` thời gian, đáp ứng follow-up, không phụ thuộc `k`.

In [22]:
from collections import Counter
from typing import List


class Solution:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        count = Counter(nums)                                  # tần suất của mỗi giá trị
        buckets = [[] for _ in range(len(nums) + 1)]           # buckets[f]: các giá trị có tần suất f
        for value, freq in count.items():
            buckets[freq].append(value)
        res = []
        for freq in range(len(nums), 0, -1):                   # từ tần suất cao xuống thấp
            res.extend(buckets[freq])
            if len(res) >= k:
                break
        return res[:k]


sol = Solution()
for nums, k, expected in examples:
    print(nums, k, "->", sol.topKFrequent(nums, k), "(kỳ vọng", expected, ")")

[1, 1, 1, 2, 2, 3] 2 -> [1, 2] (kỳ vọng [1, 2] )
[1] 1 -> [1] (kỳ vọng [1] )
[1, 2, 1, 2, 1, 2, 3, 1, 3, 2] 2 -> [1, 2] (kỳ vọng [1, 2] )


## 10. Vì sao thuật toán đúng

**Bước đếm.** `Counter(nums)[v]` đúng bằng số lần `v` xuất hiện vì mỗi phần tử được cộng đúng một lần.

**Bucket sort.**

- Với mọi giá trị `v` có tần suất `f`, `1 <= f <= n`, nên `buckets[f]` tồn tại (mảng có `n + 1` ô, chỉ số `0..n`); mỗi giá trị khác nhau được đặt vào **đúng một** thùng.
- Duyệt `freq` từ `n` xuống `1`, các giá trị được nối vào `res` theo thứ tự **tần suất không tăng**. Vì vậy sau mỗi bước, `res` chứa các giá trị có tần suất lớn nhất (theo thứ tự đó).
- Dừng ngay khi `len(res) >= k`: `k` phần tử đầu của `res` là `k` phần tử có tần suất cao nhất. Nếu một thùng làm `res` vượt `k` thì phần thừa thuộc cùng tần suất, mà đề **đảm bảo đáp án duy nhất** nên trường hợp hòa ở ranh giới không xảy ra; ngay cả khi có, cắt `res[:k]` vẫn là một đáp án hợp lệ.
- Vì `k <= u` (số giá trị khác nhau), duyệt hết thùng thì `res` có `u >= k` phần tử, vòng lặp luôn đủ `k`.

**Min-heap.** Bất biến: sau khi xử lý một số giá trị, heap chứa đúng các giá trị tốt nhất (theo tần suất) trong số đó, tối đa `k`. Khi một giá trị mới mạnh hơn đỉnh, đỉnh (yếu nhất trong nhóm) chắc chắn **không** thuộc top `k` nữa, nên loại là an toàn.

**Quickselect.** Mỗi vòng, `bigger` đều có tần suất lớn hơn mọi phần tử của `equal` và `smaller`, `equal` lớn hơn `smaller`; do đó top `k` luôn gồm một **tiền tố** của chuỗi `bigger, equal, smaller` ⇒ các nhánh `if/elif/else` chọn đúng nhóm chứa ranh giới.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `u` là số giá trị khác nhau (`u <= n`).

| Cách | Time | Space |
|---|---|---|
| 1. `list.count` từng giá trị | `O(n · u)` (xấu nhất `O(n²)`) | `O(u)` |
| 2. `Counter` + sắp xếp | `O(n + u log u)` | `O(u)` |
| 3. Min-heap cỡ `k` | `O(n + u log k)` | `O(u + k)` |
| 4. Bucket sort | `O(n)` | `O(n)` |
| 5. Quickselect | `O(n)` trung bình | `O(u)` |

Với `n` tới `10^5` và `u` tới `2 · 10^4 + 1` (giá trị trong `[-10^4, 10^4]`), cách 1 mất cỡ `2 · 10^9` phép, quá chậm; các cách còn lại chạy trong vài chục mili giây.

## 12. Bộ test

Vì đề cho **mọi thứ tự**, ta không so sánh danh sách trực tiếp mà dùng **hàm kiểm tra hợp lệ** `is_valid_topk(nums, k, res)`: kết quả có đúng `k` giá trị **khác nhau**, đều có trong `nums`, và tần suất nhỏ nhất của nhóm chọn **không nhỏ hơn** tần suất lớn nhất của nhóm không được chọn. Hàm này **độc lập** với mọi cách giải ở trên.

In [23]:
def is_valid_topk(nums, k, res):
    cnt = Counter(nums)
    if len(res) != k or len(set(res)) != k:
        return False
    if any(x not in cnt for x in res):
        return False
    chosen = set(res)
    chosen_min = min(cnt[x] for x in chosen)
    others = [cnt[v] for v in cnt if v not in chosen]
    return not others or chosen_min >= max(others)


def answer_is_unique(nums, k):
    """Đề đảm bảo: tần suất thứ k lớn hơn tần suất thứ k+1 (hoặc k = u)."""
    freqs = sorted(Counter(nums).values(), reverse=True)
    return k == len(freqs) or freqs[k - 1] > freqs[k]


sol = Solution()
impls = {
    "sort": top_k_sort,
    "heap": top_k_heap,
    "bucket": top_k_bucket,
    "quickselect": top_k_quickselect,
    "leetcode": sol.topKFrequent,
    "submit_sort": SolutionSort().topKFrequent,
    "submit_heap": SolutionHeap().topKFrequent,
    "submit_bucket": SolutionBucket().topKFrequent,
    "submit_quickselect": SolutionQuickselect().topKFrequent,
}


def check_all(nums, k, label, with_brute=True):
    for key, fn in impls.items():
        res = fn(list(nums), k)
        assert is_valid_topk(nums, k, res), (label, key, nums, k, res)
    if with_brute:
        assert is_valid_topk(nums, k, top_k_bruteforce(list(nums), k)), (label, "brute")
    print("PASS", label)


for nums, k, expected in examples:
    check_all(nums, k, f"ví dụ đề {nums[:6]}...")
    assert sorted(sol.topKFrequent(nums, k)) == sorted(expected)     # đề đảm bảo duy nhất => so được tập

PASS ví dụ đề [1, 1, 1, 2, 2, 3]...
PASS ví dụ đề [1]...
PASS ví dụ đề [1, 2, 1, 2, 1, 2]...


### 12.1. Các trường hợp biên

Một phần tử, mọi phần tử giống nhau (một thùng lớn nhất `n`), `k = u` (lấy tất cả), `k = 1`, số âm và số `0`, biên giá trị `±10^4`, tần suất hòa **không** ở ranh giới.

In [24]:
check_all([1], 1, "một phần tử")
check_all([7] * 5, 1, "mọi phần tử giống nhau (tần suất bằng n)")
check_all([1, 2, 3, 4], 4, "k = u: lấy tất cả, mọi tần suất bằng 1")
check_all([1, 1, 1, 2, 2, 3], 1, "k = 1")
check_all([-1, -1, 0, 0, 0, 5], 2, "số âm và số 0")
check_all([-10000, 10000, 10000, -10000, -10000], 1, "biên giá trị ±10^4")
check_all([4, 1, -1, 2, -1, 2, 3], 2, "ví dụ trace")
check_all([3, 3, 3, 3, 1, 1, 2], 3, "k = u = 3 (có hòa vị trí không ảnh hưởng)")
assert sol.topKFrequent([7] * 5, 1) == [7]
assert sorted(sol.topKFrequent([1, 2, 3, 4], 4)) == [1, 2, 3, 4]
print(sorted(sol.topKFrequent([-1, -1, 0, 0, 0, 5], 2)))      # [-1, 0]

PASS một phần tử
PASS mọi phần tử giống nhau (tần suất bằng n)
PASS k = u: lấy tất cả, mọi tần suất bằng 1
PASS k = 1
PASS số âm và số 0
PASS biên giá trị ±10^4
PASS ví dụ trace
PASS k = u = 3 (có hòa vị trí không ảnh hưởng)
[-1, 0]


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách "nghe hợp lý" nhưng sai: (a) **sắp theo giá trị** thay vì tần suất rồi lấy `k` đầu; (b) **lấy `k` phần tử đầu tiên khác nhau** của `nums`.

In [25]:
def wrong_by_value(nums, k):
    return sorted(set(nums), reverse=True)[:k]          # SAI: xếp theo giá trị


def wrong_first_k(nums, k):
    seen = []
    for x in nums:
        if x not in seen:
            seen.append(x)
    return seen[:k]                                     # SAI: lấy k giá trị xuất hiện đầu tiên


nums, k = [9, 5, 5, 5, 7, 7, 1], 2       # tần suất: 5:3, 7:2, 9:1, 1:1 -> đáp án {5, 7}
print("đúng:", sorted(sol.topKFrequent(nums, k)), "| theo giá trị:", wrong_by_value(nums, k), "| k đầu tiên:", wrong_first_k(nums, k))
assert not is_valid_topk(nums, k, wrong_by_value(nums, k))
assert not is_valid_topk(nums, k, wrong_first_k(nums, k))
assert is_valid_topk(nums, k, sol.topKFrequent(nums, k))

đúng: [5, 7] | theo giá trị: [9, 7] | k đầu tiên: [9, 5]


### 12.3. Đối chiếu ngẫu nhiên (có seed) và oracle độc lập

Sinh các mảng ngẫu nhiên **đúng ràng buộc** (`1 <= n`, giá trị trong `[-10^4, 10^4]`, `1 <= k <= u`), với nhiều kiểu phân bố (đều, lệch mạnh, ít giá trị). Với mỗi mảng ta kiểm tra bằng `is_valid_topk` cho **mọi** cách. Với mảng thỏa **đáp án duy nhất**, ta còn so **tập** kết quả với oracle viết riêng (sắp toàn bộ rồi cắt `k`), và so Cách 1 (vét cạn `count`) khi `n` nhỏ.

In [26]:
rng = random.Random(347)


def random_case(rng):
    style = rng.choice(["small_values", "uniform", "skewed", "few_distinct"])
    n = rng.randint(1, 60)
    if style == "small_values":
        nums = [rng.randint(-3, 3) for _ in range(n)]
    elif style == "uniform":
        nums = [rng.randint(-10000, 10000) for _ in range(n)]
    elif style == "skewed":
        nums = [min(int(rng.expovariate(0.5)), 20) - 10 for _ in range(n)]
    else:
        pool = [rng.randint(-10000, 10000) for _ in range(rng.randint(1, 4))]
        nums = [rng.choice(pool) for _ in range(n)]
    k = rng.randint(1, len(set(nums)))
    return nums, k


def oracle_unique(nums, k):
    cnt = Counter(nums)
    ranking = sorted(cnt.items(), key=lambda kv: -kv[1])        # sắp xếp độc lập
    return {v for v, _ in ranking[:k]}


valid_checked = unique_checked = 0
for _ in range(3000):
    nums, k = random_case(rng)
    check_cases = [(key, fn(list(nums), k)) for key, fn in impls.items()]
    for key, res in check_cases:
        assert is_valid_topk(nums, k, res), (key, nums, k, res)
    valid_checked += 1
    assert is_valid_topk(nums, k, top_k_bruteforce(list(nums), k))
    if answer_is_unique(nums, k):
        ref = oracle_unique(nums, k)
        for key, res in check_cases:
            assert set(res) == ref, (key, nums, k, res, ref)
        unique_checked += 1
print("PASS:", valid_checked, "mảng hợp lệ;", unique_checked, "mảng có đáp án duy nhất đã so tập với oracle")

PASS: 3000 mảng hợp lệ; 1765 mảng có đáp án duy nhất đã so tập với oracle


### 12.4. Kiểm tra hiệu năng ở giới hạn lớn nhất

`n = 10^5`, giá trị trải trong `[-10^4, 10^4]` (nhiều giá trị khác nhau, cách 1 sẽ rất chậm nên bỏ qua). Ta tạo dữ liệu có đáp án duy nhất rõ ràng (một vài giá trị "nóng" xuất hiện rất nhiều).

In [27]:
import time

rng = random.Random(3470)
big = [rng.randint(-10000, 10000) for _ in range(100000 - 3000)]
big += [123] * 1500 + [-77] * 1000 + [9999] * 500          # 3 giá trị nóng
rng.shuffle(big)
k_big = 3
assert len(big) == 100000 and answer_is_unique(big, k_big)
expected_big = {123, -77, 9999}
for key, fn in impls.items():
    t0 = time.perf_counter()
    res = fn(list(big), k_big)
    dt = time.perf_counter() - t0
    assert set(res) == expected_big and is_valid_topk(big, k_big, res), key
    print(f"{key:20s} {dt * 1000:7.1f} ms")

sort                    11.8 ms
heap                    10.7 ms
bucket                  28.8 ms
quickselect             13.1 ms


leetcode                53.6 ms
submit_sort             12.0 ms
submit_heap              9.9 ms


submit_bucket           24.6 ms
submit_quickselect      16.7 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp xếp quên `reverse=True` (lấy nhầm các giá trị **ít** nhất)

In [28]:
def sai_quen_reverse(nums, k):
    count = Counter(nums)
    return sorted(count, key=count.get)[:k]          # SAI: tăng dần -> lấy ít xuất hiện nhất


nums, k = [1, 1, 1, 2, 2, 3], 2
print("sai:", sai_quen_reverse(nums, k), "| đúng:", sorted(Solution().topKFrequent(nums, k)))
assert not is_valid_topk(nums, k, sai_quen_reverse(nums, k))

sai: [3, 2] | đúng: [1, 2]


**Vì sao sai:** `sorted` mặc định sắp **tăng dần**, nên `[:k]` là `k` giá trị **ít** xuất hiện nhất. **Cách sửa:** thêm `reverse=True` (hoặc `key=lambda x: -count[x]`, hoặc lấy `[-k:]`).

### Lỗi 2: Tạo thùng bằng `[[]] * m`

In [29]:
def sai_thung_dung_chung(nums, k):
    count = Counter(nums)
    buckets = [[]] * (len(nums) + 1)                 # SAI: mọi thùng là MỘT list
    for value, freq in count.items():
        buckets[freq].append(value)
    res = []
    for freq in range(len(nums), 0, -1):
        res.extend(buckets[freq])
        if len(res) >= k:
            break
    return res[:k]


nums, k = [3, 1, 1, 2, 2, 2], 2          # đáp án đúng {2, 1}
print("sai:", sai_thung_dung_chung(nums, k), "| đúng:", sorted(Solution().topKFrequent(nums, k)))
assert not is_valid_topk(nums, k, sai_thung_dung_chung(nums, k))

sai: [3, 1] | đúng: [1, 2]


**Vì sao sai:** `[[]] * m` sao chép **tham chiếu** tới cùng một list, nên mọi `append` đổ vào chung một danh sách; mọi thùng trông giống nhau và kết quả thành "tất cả giá trị theo thứ tự thêm vào", mất ý nghĩa tần suất. **Cách sửa:** `[[] for _ in range(m)]`.

### Lỗi 3: Đặt `(giá trị, tần suất)` thay vì `(tần suất, giá trị)` trong heap

In [30]:
def sai_heap_dao_tuple(nums, k):
    count = Counter(nums)
    heap = []
    for value, freq in count.items():
        heapq.heappush(heap, (value, freq))          # SAI: heap so sánh theo value trước
        if len(heap) > k:
            heapq.heappop(heap)                       # bỏ value nhỏ nhất, không phải tần suất nhỏ nhất
    return [value for value, _ in heap]


nums, k = [9, 5, 5, 5, 7, 7, 1], 2       # đáp án đúng {5, 7}
print("sai:", sorted(sai_heap_dao_tuple(nums, k)), "| đúng:", sorted(Solution().topKFrequent(nums, k)))
assert not is_valid_topk(nums, k, sai_heap_dao_tuple(nums, k))

sai: [7, 9] | đúng: [5, 7]


**Vì sao sai:** tuple so sánh **thành phần đầu trước**. Với `(value, freq)`, heap xếp theo **giá trị** và loại giá trị nhỏ nhất, không liên quan tới tần suất. **Cách sửa:** luôn đặt **khóa so sánh** (tần suất) ở thành phần **đầu**: `(freq, value)`.

### Lỗi 4: Mảng thùng thiếu một ô (`n` thay vì `n + 1`)

In [31]:
def sai_thung_thieu_o(nums, k):
    count = Counter(nums)
    buckets = [[] for _ in range(len(nums))]        # SAI: chỉ số 0..n-1, thiếu chỉ số n
    for value, freq in count.items():
        buckets[freq].append(value)
    return []


try:
    sai_thung_thieu_o([7, 7, 7], 1)                  # tần suất 3 = n -> buckets[3] không tồn tại
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** nếu mọi phần tử giống nhau, tần suất bằng đúng `n`, cần chỉ số `n`, tức mảng phải có `n + 1` ô (`0..n`). **Cách sửa:** `range(len(nums) + 1)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Top K** phần tử theo một tiêu chí" ⇒ nghĩ đến **đếm bằng hash map** rồi chọn bằng **heap cỡ `k`**, **bucket sort** (khi điểm là số nguyên nhỏ) hoặc **Quickselect**.
- Heap dùng để lấy **`k` lớn nhất** thì là **min-heap cỡ `k`** (đỉnh là người yếu nhất); lấy `k` **nhỏ** nhất thì dùng max-heap (đẩy số âm trong `heapq`).
- Tần suất nằm trong `[1, n]` là dấu hiệu dùng **bucket theo tần suất** (cùng mẫu với bài **Sort Characters By Frequency**, **Top K Frequent Words**).
- Trong `heapq`, đặt **khóa so sánh ở đầu tuple**; nếu cần "hòa thì xét thêm" hãy thêm thành phần thứ hai.
- `Counter.most_common(k)` là cách một dòng nhưng bên trong dùng `heapq.nlargest` (xấp xỉ `O(u log k)`): đủ cho phỏng vấn nhanh, nhưng nên hiểu cơ chế.
- Để đáp ứng follow-up "tốt hơn `O(n log n)`", nhắc tới **bucket `O(n)`** hoặc **heap `O(n log k)`**.

## 15. Biến thể của bài

1. **Top K Frequent Words (692):** `k` từ xuất hiện nhiều nhất, **hòa thì xếp theo thứ tự từ điển**.
2. **Sort Characters By Frequency (451):** sắp chuỗi theo tần suất giảm dần của từng ký tự.
3. **K Closest Points to Origin (973)** / **Kth Largest Element in an Array (215):** "top K" với tiêu chí khác (khoảng cách, giá trị) ⇒ heap hoặc Quickselect.
4. **K ít xuất hiện nhất:** đảo chiều (lấy bucket từ thấp lên).
5. **Dữ liệu dòng (stream):** mảng không vừa bộ nhớ ⇒ dùng sketch (Count-Min) / heap cập nhật dần.

In [32]:
def top_k_frequent_words(words, k):
    """Biến thể 692: tần suất giảm dần, hòa thì theo thứ tự từ điển."""
    count = Counter(words)
    return sorted(count, key=lambda w: (-count[w], w))[:k]


def frequency_sort(s):
    """Biến thể 451: sắp ký tự theo tần suất giảm dần (bucket)."""
    count = Counter(s)
    buckets = [[] for _ in range(len(s) + 1)]
    for ch, f in count.items():
        buckets[f].append(ch)
    return "".join(ch * f for f in range(len(s), 0, -1) for ch in buckets[f])


def k_least_frequent(nums, k):
    count = Counter(nums)
    return sorted(count, key=count.get)[:k]


print(top_k_frequent_words(["i", "love", "leetcode", "i", "love", "coding"], 2))     # ['i', 'love']
print(top_k_frequent_words(["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 4))
print(frequency_sort("tree"))                  # 'eert' hoặc 'eetr'
print(k_least_frequent([1, 1, 1, 2, 2, 3], 1))                              # [3]
assert top_k_frequent_words(["i", "love", "leetcode", "i", "love", "coding"], 2) == ["i", "love"]
assert frequency_sort("tree") in ("eert", "eetr")
assert sorted(frequency_sort("Aabb")) == sorted("Aabb")

['i', 'love']
['the', 'is', 'sunny', 'day']
eetr
[3]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `list.count` | đếm lại từng giá trị | `O(n · u)` | `O(u)` | chỉ để hiểu đề, `n` rất nhỏ |
| 2. `Counter` + sắp xếp | xếp tất cả theo tần suất | `O(n + u log u)` | `O(u)` | code ngắn nhất, không cần follow-up |
| 3. Min-heap cỡ `k` | giữ `k` tốt nhất | `O(n + u log k)` | `O(u + k)` | `k` nhỏ, dữ liệu dòng |
| 4. Bucket sort | thùng theo tần suất | `O(n)` | `O(n)` | **nên nộp** (follow-up, chắc chắn `O(n)`) |
| 5. Quickselect | chia theo mốc, chọn nhóm | `O(n)` trung bình | `O(u)` | khi muốn tránh thùng, thích chọn thứ hạng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Counter("banana")`, `Counter("banana").most_common(1)`, `sorted([3, 1, 2], key=lambda x: -x)`, `[[]] * 2 == [[], []]`, `Solution().topKFrequent([4, 4, 5], 1)`.

**Bài 2 (tính tay).** Với `nums = [5,5,5,4,4,3,3,3,3,2]`, `k = 2`: lập bảng tần suất, vẽ các thùng `0..10`, rồi chỉ ra kết quả.

**Bài 3 (điền TODO).** Hoàn thành `top_k_todo(nums, k)` theo Cách 4 (bucket sort).

In [33]:
def top_k_todo(nums, k):
    # TODO 1: count = Counter(nums)
    # TODO 2: tạo buckets gồm len(nums) + 1 list RIÊNG BIỆT
    # TODO 3: với mỗi (value, freq) trong count.items(): đặt value vào buckets[freq]
    # TODO 4: duyệt freq từ len(nums) xuống 1, nối buckets[freq] vào res, dừng khi len(res) >= k
    # TODO 5: trả về res[:k]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `top_k_heap2(nums, k)` theo Cách 3 (min-heap cỡ `k`) **mà không nhìn lại code**, dùng `heapq.heappush` rồi `heapq.heappop` khi heap vượt `k` phần tử (thay cho `heapreplace`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `top_k_words_ans(words, k)` cho **Top K Frequent Words** (hòa thì từ điển nhỏ hơn đứng trước) bằng **sắp xếp** và nhắc: vì sao dùng `(-tần suất, từ)` làm khóa. (b) Viết `least_frequent_k(nums, k)` lấy `k` giá trị **ít** xuất hiện nhất (đáp án duy nhất) bằng bucket (duyệt thùng từ thấp lên).

In [34]:
def top_k_heap2_todo(nums, k):
    # TODO: Counter -> với mỗi (value, freq): heappush (freq, value); nếu len(heap) > k thì heappop
    pass


def least_frequent_k_todo(nums, k):
    # TODO: bucket như Cách 4 nhưng duyệt freq từ 1 lên len(nums)
    pass

# KHÔNG gọi hàm TODO ở đây.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `Counter("banana")` đếm `a: 3`, `n: 2`, `b: 1`; `most_common(1)` là `[('a', 3)]`; `[[]] * 2 == [[], []]` là `True` (nhưng hai list **cùng là một**). Bài 3: nhớ cần `n + 1` ô và `[[] for _ in range(...)]`.
- **Mức 2 (vừa):** Bài 2: `5:3, 4:2, 3:4, 2:1` ⇒ `buckets[4] = [3]`, `buckets[3] = [5]`, `buckets[2] = [4]`, `buckets[1] = [2]`; lấy từ cao xuống: `3`, rồi `5`. Bài 4: push `(freq, value)` rồi nếu `len(heap) > k` thì `heappop` (bỏ tần suất nhỏ nhất).
- **Mức 3 (sâu):** Bài 5a: `sorted(count, key=lambda w: (-count[w], w))[:k]`; dấu `-` để tần suất giảm dần còn từ vẫn tăng dần. Bài 5b: giống bucket nhưng `for freq in range(1, len(nums) + 1)`; dừng khi đủ `k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1
assert Counter("banana") == Counter({"a": 3, "n": 2, "b": 1})
assert Counter("banana").most_common(1) == [("a", 3)]
assert sorted([3, 1, 2], key=lambda x: -x) == [3, 2, 1]
assert [[]] * 2 == [[], []]
assert Solution().topKFrequent([4, 4, 5], 1) == [4]

# Bài 2
nums2 = [5, 5, 5, 4, 4, 3, 3, 3, 3, 2]
cnt2 = Counter(nums2)
assert dict(cnt2) == {5: 3, 4: 2, 3: 4, 2: 1}
buckets2 = [[] for _ in range(len(nums2) + 1)]
for v, f in cnt2.items():
    buckets2[f].append(v)
assert buckets2[4] == [3] and buckets2[3] == [5] and buckets2[2] == [4] and buckets2[1] == [2]
assert sorted(Solution().topKFrequent(nums2, 2)) == [3, 5]

# Bài 3
def top_k_todo_ans(nums, k):
    count = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for value, freq in count.items():
        buckets[freq].append(value)
    res = []
    for freq in range(len(nums), 0, -1):
        res.extend(buckets[freq])
        if len(res) >= k:
            break
    return res[:k]

for nums, k, expected in examples:
    assert sorted(top_k_todo_ans(nums, k)) == sorted(expected)

# Bài 4
def top_k_heap2(nums, k):
    count = Counter(nums)
    heap = []
    for value, freq in count.items():
        heapq.heappush(heap, (freq, value))
        if len(heap) > k:
            heapq.heappop(heap)
    return [v for _, v in heap]

for nums, k, expected in examples:
    assert sorted(top_k_heap2(nums, k)) == sorted(expected)
rng5 = random.Random(1)
for _ in range(300):
    nums, k = random_case(rng5)
    assert is_valid_topk(nums, k, top_k_heap2(nums, k))

# Bài 5a
def top_k_words_ans(words, k):
    count = Counter(words)
    return sorted(count, key=lambda w: (-count[w], w))[:k]

assert top_k_words_ans(["i", "love", "leetcode", "i", "love", "coding"], 2) == ["i", "love"]
assert top_k_words_ans(["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 4) == ["the", "is", "sunny", "day"]

# Bài 5b
def least_frequent_k(nums, k):
    count = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for value, freq in count.items():
        buckets[freq].append(value)
    res = []
    for freq in range(1, len(nums) + 1):
        res.extend(buckets[freq])
        if len(res) >= k:
            break
    return res[:k]

assert least_frequent_k([1, 1, 1, 2, 2, 3], 1) == [3]
assert sorted(least_frequent_k([1, 1, 1, 2, 2, 3], 2)) == [2, 3]
print("Tất cả đáp án bài tập đều pass")

Tất cả đáp án bài tập đều pass


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm + bucket theo tần suất)

```python
class Solution:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        count = Counter(nums)
        buckets = [[] for _ in range(len(nums) + 1)]
        for value, freq in count.items():
            buckets[freq].append(value)
        res = []
        for freq in range(len(nums), 0, -1):
            res.extend(buckets[freq])
            if len(res) >= k:
                break
        return res[:k]
```

**Template heap** (khi `k` nhỏ)

```python
count = Counter(nums)
heap = []
for value, freq in count.items():
    heapq.heappush(heap, (freq, value))
    if len(heap) > k:
        heapq.heappop(heap)
return [v for _, v in heap]
```

**Checklist**

- [ ] Đã đếm tần suất **một lần** (`Counter`/`dict`) thay vì `count` lặp lại?
- [ ] Sắp xếp/duyệt đúng **chiều** (tần suất giảm dần, hoặc min-heap cỡ `k`)?
- [ ] Heap lưu `(tần suất, giá trị)` với tần suất ở **đầu** tuple?
- [ ] Bucket có `len(nums) + 1` ô và được tạo bằng `[[] for _ in range(...)]`?
- [ ] Trả về **giá trị** (không phải tần suất, không phải chỉ số) và đúng `k` phần tử?
- [ ] Đã thử `nums = [1]`, mọi phần tử giống nhau, `k = số giá trị khác nhau`, số âm?
- [ ] Skeleton đúng `class Solution` / `def topKFrequent(self, nums: List[int], k: int) -> List[int]`?